In [6]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

# ---------- 0) 헬퍼 ----------
def hungarian_align(y_pred: np.ndarray, y_true: np.ndarray):
    """
    y_pred (N,), y_true (N,) -> (y_aligned, mapping)
    mapping: {pred_cluster -> true_label}
    """
    Kp = int(y_pred.max()) + 1
    Kt = int(y_true.max()) + 1
    K = max(Kp, Kt)

    # contingency matrix (K x K): count of (pred=k, true=j)
    M = np.zeros((K, K), dtype=np.int64)
    for k in range(Kp):
        for j in range(Kt):
            M[k, j] = np.sum((y_pred == k) & (y_true == j))

    # maximize matches -> minimize negative counts
    cost = -(M[:Kp, :Kt])
    r, c = linear_sum_assignment(cost)
    mapping = {int(rr): int(cc) for rr, cc in zip(r, c)}

    y_aligned = np.array([mapping.get(int(k), int(k)) for k in y_pred], dtype=int)
    return y_aligned, mapping, M

def miscls_count(y1, y2):
    return int(np.sum(y1 != y2))

# ---------- 1) 로드 ----------
npz_path = "/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/text-embedding-3-small/soft_results.npz"
csv_path = "/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/cluster_20ng.csv"

data = np.load(npz_path, allow_pickle=True)
P = data["P"]                # (N,K)
C = data["C"]
y_pred_hard = data["y_pred_hard"]         # (N,)
y_pred_aligned_saved = data["y_pred_aligned"]  # (N,)
mapping_arr = data["mapping"]
mapping_saved = {int(k): int(v) for k, v in mapping_arr}

df = pd.read_csv(csv_path)

# ---------- 2) 순서 맞추기 ----------
# CSV에 'index'가 있으면 그걸 기준으로 정렬해 npz 순서와 맞춘다 (일반적으로 필요)
if "index" in df.columns:
    df = df.sort_values("index").reset_index(drop=True)

y_true = df["label_id"].to_numpy()
N = P.shape[0]

assert len(y_true) == N, f"Length mismatch: y_true={len(y_true)} vs P={N}"

# ---------- 3) 필요 시 argmax 재계산(검증용) ----------
y_pred_hard_fromP = np.argmax(P, axis=1)
if not np.array_equal(y_pred_hard_fromP, y_pred_hard):
    print("[warn] y_pred_hard != argmax(P). Using argmax(P) to be safe.")
    y_pred_hard = y_pred_hard_fromP

# ---------- 4) 전후 misclassified 비교 ----------
mis_before = miscls_count(y_pred_hard, y_true)
mis_after_saved = miscls_count(y_pred_aligned_saved, y_true)

print(f"[BEFORE] misclassified (hard vs y_true):  {mis_before}")
print(f"[AFTER - saved] misclassified (aligned_saved vs y_true): {mis_after_saved}")

# ---------- 5) 현재 y_true에 대해 '다시' 헝가리안 수행 ----------
y_pred_aligned_new, mapping_new, M = hungarian_align(y_pred_hard, y_true)
mis_after_new = miscls_count(y_pred_aligned_new, y_true)

print(f"[AFTER - recomputed] misclassified (aligned_new vs y_true): {mis_after_new}")

# 변화가 없을 때 진단 힌트
if mis_after_new == mis_before:
    print("[note] Misclassified count did not change after Hungarian.")
    print("Possible causes: (1) ordering mismatch, (2) gt labels misaligned, (3) degenerate contingency.")
    # contingency 매트릭스 요약
    top_hits = M.max(axis=1)
    print(f"[diag] per-cluster max hits (first 10): {top_hits[:10]}")

# ---------- 6) 저장(덮어쓰기) ----------
np.savez(
    npz_path,
    P=P,
    C=C,
    y_pred_hard=y_pred_hard,
    y_pred_aligned=y_pred_aligned_new,          # 새로 계산한 정렬 결과로 업데이트
    mapping=list(mapping_new.items())           # 새 매핑 저장
)
print("[Saved] soft_results.npz updated with recomputed Hungarian alignment.")


[warn] y_pred_hard != argmax(P). Using argmax(P) to be safe.
[BEFORE] misclassified (hard vs y_true):  17020
[AFTER - saved] misclassified (aligned_saved vs y_true): 7464
[AFTER - recomputed] misclassified (aligned_new vs y_true): 7464
[Saved] soft_results.npz updated with recomputed Hungarian alignment.
